In [ ]:
"""
- w2_j_ctc_by_student_lv7.0.csv     (國文教師對班級評量)
- w2_j_etc_by_student_lv7.0.csv     (英文教師對班級評量)
- w2_j_mtc_by_student_lv7.0.csv     (數學教師對班級評量)
- w2_j_sev_lv7.0.csv                (學生表現評量)

輸出：
- cleaned_merged_w2_hw_sev_sorted.csv
- field_dictionary.csv
"""
import pandas as pd
import numpy as np
import re
from pathlib import Path

files = {
    "chi": ["w2_j_ctc_by_student_lv7.0.csv"], #國文教師對班級評量
    "eng": ["w2_j_etc_by_student_lv7.0.csv"], #英文教師對班級評量
    "math": ["w2_j_mtc_by_student_lv7.0.csv"], #數學教師對班級評量
    "sev": ["w2_j_sev_lv7.0.csv"] #學生表現評量
}

ENCODINGS = ["utf-8", "utf-8-sig", "cp950", "big5"]

def find_and_read(candidates):
    for c in candidates:
        p = Path(c)
        if p.exists():
            for enc in ENCODINGS:
                try:
                    df = pd.read_csv(p, encoding=enc)
                    print(f"Loaded {p} (encoding={enc}) shape={df.shape}")
                    return df
                except Exception as e:
                    # print(f"Failed {p} with enc {enc}: {e}")
                    continue
    raise FileNotFoundError(f"No file found among candidates: {candidates}")

# 讀檔
df_chi = find_and_read(files["chi"]) #國文教師對班級評量
df_eng = find_and_read(files["eng"]) #英文教師對班級評量
df_math = find_and_read(files["math"]) #數學教師對班級評量
df_sev = find_and_read(files["sev"]) #學生表現評量

# ------------------------------
# 欄位標準化：小寫化並去頭尾空白
# ------------------------------
def normalize_columns(df):
    df = df.rename(columns=lambda c: str(c).strip().lower())
    return df

df_chi = normalize_columns(df_chi)
df_eng = normalize_columns(df_eng)
df_math = normalize_columns(df_math)
df_sev = normalize_columns(df_sev)

# ------------------------------
# stud_id 正規化成純數字（int），並建立合併用欄位 stud_id_merge 與學校代碼 w2sch_id_merge
# - 例如 't00001' -> 1, '001' -> 1
# - 如果沒有 digits，會變成 NaN
# ------------------------------
def extract_digits_or_nan(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    digits = re.findall(r'\d+', s)
    if not digits:
        return np.nan
    return int(''.join(digits))

for df in [df_chi, df_eng, df_math, df_sev]:
    if "stud_id" not in df.columns:
        raise KeyError("所有輸入檔必須含有 'stud_id' 欄位。")
    df["stud_id_norm"] = df["stud_id"].apply(extract_digits_or_nan)
    # 學校代碼可能是 w2sch_id；若不存在則為 NaN
    if "w2sch_id" in df.columns:
        df["w2sch_id_norm"] = df["w2sch_id"].apply(lambda x: int(''.join(re.findall(r'\d+', str(x)))) if pd.notna(x) and re.findall(r'\d+', str(x)) else np.nan)
    else:
        df["w2sch_id_norm"] = np.nan

    # 用於合併的 string 欄位（但這裡我們也保留 numeric 以便排序）
    df["stud_id_merge"] = df["stud_id_norm"]
    df["w2sch_id_merge"] = df["w2sch_id_norm"]

# ------------------------------
# 找出每科作業相關欄位（頻率與 25~30 類型）
# 用 regex 在各 dataframe 中尋找可能的欄名
# ------------------------------
def locate_hw_cols(df, subject_prefixes):
    """
    subject_prefixes: list of strings e.g. ["w2ctc", "w2xtc", "ctc"]
    返回 dict: {"freq": colname or None, 25: colname or None, ... 30: colname or None}
    """
    found = {}
    cols = df.columns.tolist()
    # freq patterns
    freq_candidates = []
    for p in subject_prefixes:
        freq_candidates += [p + "22a", p + "22", p + "_22a", p + "_22"]
    # find freq
    freq_col = None
    for c in cols:
        lc = c.lower()
        for pat in freq_candidates:
            if pat in lc:
                freq_col = c
                break
        if freq_col:
            break
    found["freq"] = freq_col

    for i in range(25,30):
        qcol = None
        for c in cols:
            lc = c.lower()
            if any(p in lc and str(i) in lc for p in subject_prefixes):
                qcol = c
                break
            if "w2" in lc and str(i) in lc:
                qcol = c
                break
            if lc == str(i):
                qcol = c
                break
        found[i] = qcol
    return found

subjects = {
    "chi": ["w2ctc", "w2xtc", "ctc"],
    "eng": ["w2etc", "w2xtc", "etc"],
    "mat": ["w2mtc", "w2xtc", "mtc"]
}

cols_ch = locate_hw_cols(df_chi, subjects["chi"])
cols_en = locate_hw_cols(df_eng, subjects["eng"])
cols_mt = locate_hw_cols(df_math, subjects["mat"])

print("Located columns (summary):")
print("CH:", cols_ch)
print("EN:", cols_en)
print("MT:", cols_mt)

# ------------------------------
# 映射設定（可以視需求調整）
# ------------------------------
hw_freq_map = {1:1.5, 2:3.5, 3:5.5, 4:7.5, 5:9.0, 6:0.0}
hw_type_map = {1:1.0, 2:0.66, 3:0.33, 4:0.0, 5:np.nan}

# ------------------------------
# 將找到的欄位標準化成我們的欄名，並 map 成數值
# 產生的欄位會以前綴 chi_/eng_/mat_ 開頭
# ------------------------------
def apply_hw_mapping(df, cols_loc, subj_tag):
    # freq
    freq_c = cols_loc.get("freq")
    if freq_c and freq_c in df.columns:
        df[f"{subj_tag}_hw_freq_raw"] = pd.to_numeric(df[freq_c], errors='coerce')
        df[f"{subj_tag}_hw_freq_num"] = df[f"{subj_tag}_hw_freq_raw"].map(hw_freq_map)
    else:
        df[f"{subj_tag}_hw_freq_raw"] = np.nan
        df[f"{subj_tag}_hw_freq_num"] = np.nan

    # q25~q29
    for i in range(25,30):
        c = cols_loc.get(i)
        if c and c in df.columns:
            df[f"{subj_tag}_q{i}_raw"] = pd.to_numeric(df[c], errors='coerce')
            df[f"{subj_tag}_q{i}_num"] = df[f"{subj_tag}_q{i}_raw"].map(hw_type_map)
        else:
            df[f"{subj_tag}_q{i}_raw"] = np.nan
            df[f"{subj_tag}_q{i}_num"] = np.nan

    # count how many q25~q29 are '經常' (raw==1)
    q_raw_cols = [f"{subj_tag}_q{i}_raw" for i in range(25,30)]
    df[f"{subj_tag}_num_frequent_types"] = df[q_raw_cols].eq(1).sum(axis=1, skipna=True)
    return df

df_chi = apply_hw_mapping(df_chi, cols_ch, "chi")
df_eng = apply_hw_mapping(df_eng, cols_en, "eng")
df_math = apply_hw_mapping(df_math, cols_mt, "mat")

# ------------------------------
# 選取合併需要的欄位（只保留標準化產生的欄位 + 一些原始可能重要欄）
# ------------------------------
def select_for_merge(df, subj_tag):
    keep = ["stud_id_merge", "w2sch_id_merge"]

    keep += [c for c in df.columns if c.startswith(f"{subj_tag}_") and (c.endswith("_num") or c.endswith("_raw") or c.endswith("_num_frequent_types"))]
    
    for score in ["w2nright","w2cfree","w2math"]:
        if score in df.columns:
            keep.append(score)
    
    for c in ["w2ctch_id", "w2etch_id", "w2mtch_id", "w2chin_id", "w2engl_id", "w2math_id", "w2cls_id"]:
        if c in df.columns:
            keep.append(c)
    keep = [k for k in keep if k in df.columns]
    return df[keep].copy()

chi_sel = select_for_merge(df_chi, "chi")
eng_sel = select_for_merge(df_eng, "eng")
mat_sel = select_for_merge(df_math, "mat")
sev_sel = df_sev.copy()

# ------------------------------
# 合併：以 stud_id_merge, w2sch_id_merge 為 key（outer merge 保留全部學生）
# ------------------------------
merged = chi_sel.merge(eng_sel, on=["stud_id_merge","w2sch_id_merge"], how="outer", suffixes=("_chi","_eng"))
merged = merged.merge(mat_sel, on=["stud_id_merge","w2sch_id_merge"], how="outer")
merged = merged.merge(sev_sel, on=["stud_id_merge","w2sch_id_merge"], how="outer", suffixes=("","_sev"))

print("Merged shape:", merged.shape)

# ------------------------------
# 衍生欄位：hw_freq_mean_all、unified scores
# ------------------------------
freq_cols = [c for c in merged.columns if c.endswith("_hw_freq_num")]
merged["hw_freq_mean_all"] = merged[freq_cols].mean(axis=1, skipna=True)

for score in ["w2nright","w2cfree","w2math"]:
    matches = [c for c in merged.columns if c == score or c.endswith("_"+score) or score in c]
    if matches:
        merged[f"{score}_unified"] = pd.to_numeric(merged[matches[0]], errors='coerce')
    else:
        merged[f"{score}_unified"] = np.nan

# 處理學生表現 sev 欄：5 -> NaN (不清楚)
sev_cols = ["w2tcs1","w2tcs2","w2tcs3","w2tes1","w2tes2","w2tes3","w2tms1","w2tms2","w2tms3"]
for c in sev_cols:
    if c in merged.columns:
        merged[c] = pd.to_numeric(merged[c], errors='coerce')
        merged[c] = merged[c].replace({5: np.nan})

# ------------------------------
# 重要：把 stud_id_merge 轉為整數（若可能），並以數值排序
# - 若 stud_id_merge 為 NaN 的行仍保留在最後
# - 先建立一個整數欄 stud_id_sorted，供排序與輸出
# ------------------------------

merged["stud_id_sorted"] = pd.to_numeric(merged["stud_id_merge"], errors='coerce').astype("Int64")

merged["w2sch_id_sorted"] = pd.to_numeric(merged["w2sch_id_merge"], errors='coerce').astype("Int64")

merged = merged.sort_values(by=["w2sch_id_sorted","stud_id_sorted"], na_position='last').reset_index(drop=True)

merged["stud_id_output"] = merged["stud_id_sorted"].astype("Int64")

# ------------------------------
# 輸出清洗後資料（含排序）
# ------------------------------
out_path = Path("cleaned_merged.csv")

ordered_cols = ["stud_id_output", "w2sch_id_sorted"] + [c for c in merged.columns if c not in ("stud_id_output","w2sch_id_sorted")]
merged.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved cleaned+sorted merged file to:", out_path)

# ------------------------------
# 自動產生欄位字典（field dictionary）
# 會嘗試為 merged 的常見欄位寫上來源與說明
# ------------------------------
desc_map = {
    # identifiers
    "stud_id_output": ("derived", "學生代碼（已正規化為整數，用於排序與輸出）"),
    "w2sch_id_sorted": ("derived", "學校代碼（數值化，用於排序）"),
    # hw freq raw/num (example for patterns)
    "chi_hw_freq_raw": ("w2_j_ctc_by_student_lv7.0.csv", "國文教師原始作業頻率欄位（原始選項）"),
    "chi_hw_freq_num": ("derived", "國文作業每週次數（選項映射為近似次數，例如 1->1.5）"),
    "eng_hw_freq_raw": ("w2_j_etc_by_student lv7.0.csv", "英文教師原始作業頻率欄位（原始選項）"),
    "eng_hw_freq_num": ("derived", "英文作業每週次數（映射數值）"),
    "mat_hw_freq_raw": ("w2_j_mtc_by_student_lv7.0.csv", "數學教師原始作業頻率欄位（原始選項）"),
    "mat_hw_freq_num": ("derived", "數學作業每週次數（映射數值）"),
    # hw type counts
    "chi_num_frequent_types": ("derived", "國文中被標為「經常」(raw==1) 的作業類型數量（25~29）"),
    "eng_num_frequent_types": ("derived", "英文中被標為「經常」的作業類型數量（25~29）"),
    "mat_num_frequent_types": ("derived", "數學中被標為「經常」的作業類型數量（25~29）"),
    "hw_freq_mean_all": ("derived", "三科作業頻率平均（取可用科目的平均值）"),
    # unified scores
    "w2nright_unified": ("derived", "綜合分析能力測驗答對題數（若存在來源欄則統一到此欄）"),
    "w2cfree_unified": ("derived", "一般分析能力測驗答對題數（統一欄）"),
    "w2math_unified": ("derived", "數學分析能力測驗答對題數（統一欄）"),
    # student evaluation (sev)
    "w2tcs1": ("w2_j_sev_lv7.0.csv", "國文－跟得上進度嗎？（1=進度超前~4=完全跟不上，5=不清楚->已轉NaN）"),
    "w2tcs2": ("w2_j_sev_lv7.0.csv", "國文－用功程度（1非常用功~4非常不用功，5->NaN）"),
    "w2tcs3": ("w2_j_sev_lv7.0.csv", "國文－作業表現（1總是遲交~4從未遲交，5->NaN）"),
    # 其他科目以此類推...
}

# 建立欄位字典 DataFrame：若欄位在 desc_map 則取描述，否則自動產生基本說明
field_rows = []
for col in merged.columns:
    if col in desc_map:
        src, desc = desc_map[col]
    else:
        # 自動生成簡短描述
        if col.endswith("_hw_freq_num"):
            src = "teacher_eval"
            desc = "某科作業每週次數（由原始選項映射）"
        elif re.search(r"_q\d+_num$", col):
            src = "teacher_eval"
            desc = "作業類型頻率數值（25~29 題對應的類型，映射後）"
        elif col.endswith("_num_frequent_types"):
            src = "derived"
            desc = "被標示為經常的作業類型數量（25~29）"
        elif col.endswith("_unified"):
            src = "derived"
            desc = "測驗分數統一欄位（若原始資料存在則填入）"
        elif col in ["stud_id_merge","w2sch_id_merge","stud_id_norm","w2sch_id_norm","stud_id_sorted","w2sch_id_sorted"]:
            src = "internal"
            desc = "internal id used for merging/sorting"
        else:
            src = "unknown"
            desc = "自動生成欄位，請檢查原始欄位來源以補充說明"
    field_rows.append({"column": col, "source": src, "description": desc})

field_df = pd.DataFrame(field_rows)
field_df.to_csv("field_dictionary.csv", index=False, encoding="utf-8-sig")
print("Saved field dictionary to: field_dictionary.csv")

# ------------------------------
# 最後顯示輸出摘要
# ------------------------------
print("\nOutput files:")
print(" - cleaned (sorted):", out_path)
print(" - field dictionary:", Path("field_dictionary.csv").resolve())
print("\nTop rows of final cleaned data (first 8 rows):")
print(merged.head(8)[["stud_id_output","w2sch_id_sorted"] + [c for c in merged.columns if c.endswith("_hw_freq_num")]].to_string(index=False))


Loaded w2_j_ctc_by_student_lv7.0.csv (encoding=utf-8) shape=(17284, 70)
Loaded w2_j_etc_by_student_lv7.0.csv (encoding=utf-8) shape=(16942, 69)
Loaded w2_j_mtc_by_student_lv7.0.csv (encoding=utf-8) shape=(16882, 70)
Loaded w2_j_sev_lv7.0.csv (encoding=utf-8) shape=(17958, 35)
Located columns (summary):
CH: {'freq': 'w2ctc22a', 25: 'w2ctc25', 26: 'w2ctc26', 27: 'w2ctc27', 28: 'w2ctc28', 29: 'w2ctc29'}
EN: {'freq': 'w2etc22a', 25: 'w2etc25', 26: 'w2etc26', 27: 'w2etc27', 28: 'w2etc28', 29: 'w2etc29'}
MT: {'freq': 'w2mtc22a', 25: 'w2mtc25', 26: 'w2mtc26', 27: 'w2mtc27', 28: 'w2mtc28', 29: 'w2mtc29'}
Merged shape: (17962, 93)
Saved cleaned+sorted merged file to: cleaned_merged.csv
Saved field dictionary to: field_dictionary.csv

Output files:
 - cleaned (sorted): cleaned_merged.csv
 - field dictionary: C:\Users\zxzy2\OneDrive\Desktop\code\NTUT\PBL\project\field_dictionary.csv

Top rows of final cleaned data (first 8 rows):
 stud_id_output  w2sch_id_sorted  chi_hw_freq_num  eng_hw_freq_num 

In [2]:
import pandas as pd
import re

df = pd.read_csv("cleaned_merged.csv", encoding="utf-8-sig", low_memory=False)

# 所有要刪掉的欄位（*4, *5）
drop_patterns = [
    r"w2tcs4$", r"w2tcs5$",
    r"w2tes4$", r"w2tes5$",
    r"w2tms4$", r"w2tms5$",
    r"w2td01$", r"w2td02$",
    r"w2sumerr", r"w2summis",
]

drop_cols = []

# 找出所有符合 pattern 的欄位
for pat in drop_patterns:
    for col in df.columns:
        if re.match(pat, col, flags=re.IGNORECASE):
            drop_cols.append(col)

print("將移除欄位:", drop_cols)

# w2td01 ~ w2td15
drop_cols2 = [col for col in df.columns if re.match(r"w2td(0[1-9]|1[0-5])$", col, flags=re.IGNORECASE)]

for i in drop_cols2:
    drop_cols.append(i)

print("將移除欄位:", drop_cols2)

df2 = df.drop(columns=drop_cols)

df2.to_csv("cleaned_merged.csv",
           index=False, encoding="utf-8-sig")

print("已輸出 trimmed 檔案：cleaned_merged.csv")


將移除欄位: ['w2tcs4', 'w2tcs5', 'w2tes4', 'w2tes5', 'w2tms4', 'w2tms5', 'w2td01', 'w2td02', 'w2sumerr', 'w2summis']
將移除欄位: ['w2td01', 'w2td02', 'w2td03', 'w2td04', 'w2td05', 'w2td06', 'w2td07', 'w2td08', 'w2td09', 'w2td10', 'w2td11', 'w2td12', 'w2td13', 'w2td14', 'w2td15']
已輸出 trimmed 檔案：cleaned_merged.csv


In [4]:
# 方案B：保留必要欄位 + 部分資訊欄位（適中）
import pandas as pd
import re
from pathlib import Path

in_path = Path("cleaned_merged.csv")
if not in_path.exists():
    raise FileNotFoundError(f"找不到輸入檔：{in_path}. 請確認檔案名稱與路徑。")

df = pd.read_csv(in_path, encoding="utf-8-sig", low_memory=False)

# ---------- 1) 決定要保留的欄位清單（動態偵測存在欄位） ----------
keep = []

# --- A. 學生 / 學校識別欄（優先使用已正規化欄位） ---
# 優先順序： stud_id_output -> stud_id_sorted -> stud_id_merge -> stud_id (若為數字) -> stud_id_norm
id_candidates = ["stud_id_output", "stud_id_sorted", "stud_id_merge", "stud_id", "stud_id_norm"]
for c in id_candidates:
    if c in df.columns:
        keep.append(c)
        break

sch_candidates = ["w2sch_id_sorted", "w2sch_id_merge", "w2sch_id", "w2sch_id_norm"]
for c in sch_candidates:
    if c in df.columns:
        keep.append(c)
        break

# --- B. 老師 / 班級 id（若存在，保留以便後續分層分析） ---
for c in ["w2ctch_id","w2etch_id","w2mtch_id","w2chin_id","w2engl_id","w2math_id","w2cls_id"]:
    if c in df.columns:
        keep.append(c)

# --- C. 作業頻率欄（raw 與 numeric）與其平均（各科） ---
for subj_tag in ["chi","eng","mat","w2ctc","w2etc","w2mtc"]:
    # 常見命名： chi_hw_freq_raw, chi_hw_freq_num  或 w2ctc_hw_freq_num
    # 只要欄名包含這些片段就保留
    for col in df.columns:
        low = col.lower()
        if any(p in low for p in [f"{subj_tag}_hw_freq_raw", f"{subj_tag}_hw_freq_num", f"{subj_tag}_hw_freq"]):
            keep.append(col)

# 另外保留三科平均（若存在）
for c in ["hw_freq_mean_all"]:
    if c in df.columns:
        keep.append(c)

# --- D. 作業類型 25~30 的 numeric 欄位（各科） ---
for subj in ["chi","eng","mat","w2ctc","w2etc","w2mtc"]:
    for i in range(25,31):
        pat_num = f"{subj}_q{i}_num"
        pat_raw = f"{subj}_q{i}_raw"
        # 可能命名有變化，採 contains 搜尋
        for col in df.columns:
            low = col.lower()
            if pat_num in low or pat_raw in low or (f"q{i}" in low and subj in low):
                keep.append(col)

# --- E. 每科被標示為「經常」的作業類型數（derived） ---
for col in df.columns:
    if re.search(r"(chi|eng|mat|w2ctc|w2etc|w2mtc).*num_frequent_types", col, flags=re.IGNORECASE):
        keep.append(col)

# --- F. 測驗分數（只保留 unified 欄位） ---
for score in ["w2nright_unified","w2cfree_unified","w2math_unified"]:
    if score in df.columns:
        keep.append(score)
# 若找不到 unified 欄位，再退而求其次保留任何含有這些名字的欄位（最後備援）
if not any(c in df.columns for c in ["w2nright_unified","w2cfree_unified","w2math_unified"]):
    for col in df.columns:
        low = col.lower()
        if "w2nright" in low or "w2cfree" in low or "w2math" in low:
            # 但避免保留太多重複欄位：只選第一個 match
            if "w2nright" in low and "w2nright_unified" not in keep:
                keep.append(col)
            if "w2cfree" in low and "w2cfree_unified" not in keep:
                keep.append(col)
            if "w2math" in low and "w2math_unified" not in keep:
                keep.append(col)

# --- G. 學生表現評量 sev（國/英/數 1~3） ---
for prefix in ["w2tcs","w2tes","w2tms"]:
    for i in [1,2,3]:
        col = f"{prefix}{i}"
        if col in df.columns:
            keep.append(col)

# --- H. 其它你可能想保留的輔助欄位（例如原始 raw 欄位、合併用內部欄的部分） ---
# 但一般不保留大量 internal 欄位，這裡不主動加

# ---------- 2) 移除重複並確保欄位存在 ----------
# 保留順序並去重
keep_ordered = []
for c in keep:
    if c not in keep_ordered and c in df.columns:
        keep_ordered.append(c)

# 若找不到任何核心 ID（理論上不會發生），則 raise
if not any(k for k in keep_ordered if re.search(r"stud_id", str(k), flags=re.IGNORECASE)):
    raise RuntimeError("未在資料中找到可用的學生識別欄位（stud_id_output / stud_id 等）。請確認檔案。")

# ---------- 3) 產生 trimmed DataFrame ----------
df_trim = df[keep_ordered].copy()

# ---------- 4) 將 stud_id 欄重新命名為統一名稱 'stud_id' (選擇最合適的欄) ----------
# 如果存在 stud_id_output 就改為 stud_id；否則把第一個包含 stud_id 的欄位改名
stud_cols = [c for c in df_trim.columns if re.search(r"stud_id", c, flags=re.IGNORECASE)]
if stud_cols:
    chosen = stud_cols[0]
    # 若 chosen 不是剛好 'stud_id'，則改名
    if chosen != "stud_id":
        df_trim = df_trim.rename(columns={chosen: "stud_id"})
# 同理處理 w2sch_id
sch_cols = [c for c in df_trim.columns if re.search(r"w2sch_id", c, flags=re.IGNORECASE)]
if sch_cols:
    chosen_s = sch_cols[0]
    if chosen_s != "w2sch_id":
        df_trim = df_trim.rename(columns={chosen_s: "w2sch_id"})

# ---------- 5) 儲存結果 ----------
out_path = Path("cleaned_merged.csv")
df_trim.to_csv(out_path, index=False, encoding="utf-8-sig")

# ---------- 6) 顯示輸出摘要 ----------
print("輸出檔案：", out_path)
print("保留的欄位 (共 %d 個):" % len(df_trim.columns))
for c in df_trim.columns:
    print(" -", c)

print("\n前 5 列預覽：")
print(df_trim.head(5).to_string(index=False))


輸出檔案： cleaned_merged.csv
保留的欄位 (共 61 個):
 - stud_id
 - w2sch_id
 - w2ctch_id
 - w2etch_id
 - w2mtch_id
 - w2chin_id
 - w2engl_id
 - w2math_id
 - w2cls_id
 - chi_hw_freq_raw
 - chi_hw_freq_num
 - eng_hw_freq_raw
 - eng_hw_freq_num
 - mat_hw_freq_raw
 - mat_hw_freq_num
 - hw_freq_mean_all
 - chi_q25_raw
 - chi_q25_num
 - chi_q26_raw
 - chi_q26_num
 - chi_q27_raw
 - chi_q27_num
 - chi_q28_raw
 - chi_q28_num
 - chi_q29_raw
 - chi_q29_num
 - eng_q25_raw
 - eng_q25_num
 - eng_q26_raw
 - eng_q26_num
 - eng_q27_raw
 - eng_q27_num
 - eng_q28_raw
 - eng_q28_num
 - eng_q29_raw
 - eng_q29_num
 - mat_q25_raw
 - mat_q25_num
 - mat_q26_raw
 - mat_q26_num
 - mat_q27_raw
 - mat_q27_num
 - mat_q28_raw
 - mat_q28_num
 - mat_q29_raw
 - mat_q29_num
 - chi_num_frequent_types
 - eng_num_frequent_types
 - mat_num_frequent_types
 - w2nright_unified
 - w2cfree_unified
 - w2math_unified
 - w2tcs1
 - w2tcs2
 - w2tcs3
 - w2tes1
 - w2tes2
 - w2tes3
 - w2tms1
 - w2tms2
 - w2tms3

前 5 列預覽：
 stud_id  w2sch_id  w2ctch_